Stages in Fine-tuning a frontier model with OpenAI
1. Create dataset in jsonl format and upload to OpenAI.
2. Run training.
3. Evaluate, tweak and repeat.

In [ ]:
import os
import json
from openai import OpenAI
from datasets import load_dataset
from dotenv import load_dotenv
from huggingface_hub import login
load_dotenv(override=True)

In [ ]:
hf_token = os.getenv("HF_TOKEN")
openai_api_key = os.getenv("OPENAI_API_KEY")

if not hf_token:
    raise ValueError("Missing HF_TOKEN environment variable.")

if not openai_api_key:
    raise ValueError("Missing OPENAI_API_KEY environment variable.")

login(hf_token, add_to_git_credential=True)
openai = OpenAI(api_key=openai_api_key)
datasets = load_dataset("ed-donner/items_lite")

In [ ]:
training_data = datasets['train']
validation_data = datasets['validation']
test_data = datasets['test']

In [ ]:
SYSTEM_PROMPT = """
You are an expert product pricing model. Given any product information (title, description, specs, features, condition, brand, etc.), estimate the product’s price in USD.

Guidelines:
1. Use all provided details to infer a realistic market price.
2. If details are missing, partial, or noisy, infer from comparable products and typical pricing patterns.
3. Consider factors like brand, quality, category, materials, technical specs, and product condition.
4. Output only the estimated price as a number (example: 49.99).
5. Do not include "$", commas, labels, explanations, or extra words.
6. Return exactly one raw floating-point value.
"""

USER_PROMPT = "Estimate the price of this product. Respond with the price, no explanation" 

In [ ]:
def generate_message_for(item, stage='training'):
    """Build chat messages for training or inference.

    Args:
        item (dict): Product example containing at least `summary` and,
            for training, `price`.
        stage (str): Message-generation stage. Use "training" to include
            the assistant target price, otherwise user/system messages only.

    Returns:
        list[dict[str, str]]: Ordered chat messages for the model.
    """
    if stage == 'training':
        return [
            {"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user", "content": f"{USER_PROMPT}\n\n{item['summary']}"},
            {"role": "assistant", "content": f"Price is ${item['price']:.2f}"}
        ]
    else:
        return [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": f"{USER_PROMPT}\n\n{item['summary']}"},
    ]

In [ ]:
def make_jsonl(items):
    """Convert dataset items into OpenAI fine-tuning JSONL text.

    Args:
        items (dict | iterable[dict]): Either a Hugging Face sliced dataset
            structure (column-wise dict of lists) or an iterable of row dicts.

    Returns:
        str: Newline-delimited JSON where each line has a `messages` payload.
    """
    lines = []

    # HF Dataset slice case: dict of columns, e.g. {"summary": [...], "price": [...]}
    if isinstance(items, dict):
        records = [dict(zip(items.keys(), values)) for values in zip(*items.values())]
    else:
        # HF Dataset (iterable rows) or list of dicts
        records = list(items)

    for item in records:
        messages = generate_message_for(item, "training")
        lines.append(json.dumps({"messages": messages}))

    return "\n".join(lines)


def write_jsonl(items, filename):
    """Write JSONL training data to disk.

    Args:
        items (dict | iterable[dict]): Dataset slice or iterable of examples.
        filename (str): Output file path.

    Returns:
        None: This function writes data as a side effect.
    """
    jsonl_text = make_jsonl(items)
    with open(filename, "w", encoding="utf-8") as f:
        f.write(jsonl_text)

write_jsonl(training_data[:100], "fine_tune_train.jsonl")
write_jsonl(training_data[:20], "fine_tune_validation.jsonl")

In [ ]:
with open("fine_tune_train.jsonl", "rb") as f:
    training_file = openai.files.create(file=f, purpose="fine-tune")

with open("fine_tune_validation.jsonl", "rb") as f:
    validation_file = openai.files.create(file=f, purpose="fine-tune")

In [ ]:
fine_tune_job = openai.fine_tuning.jobs.create(
    training_file=training_file.id,
    validation_file=validation_file.id,
    model="gpt-4.1-nano-2025-04-14",
    seed=42,
    hyperparameters={"n_epochs": 2, "batch_size": 1},
    suffix="pricer"
)

job_id = fine_tune_job.id
fine_tuned_model_name = openai.fine_tuning.jobs.retrieve(job_id).fine_tuned_model


In [ ]:
def gpt_price_estimator(item):
    """Estimate price using the fine-tuned chat model.

    Args:
        item (dict): Product example containing a `summary` field.

    Returns:
        str: Model-predicted price text returned by the chat completion API.
    """
    response = openai.chat.completions.create(
        model=fine_tuned_model_name,
        messages=generate_message_for(item, 'inference'),
        max_tokens=8
    )
    price = response.choices[0].message.content
    return price

In [ ]:
gpt_price_estimator(test_data[0])